<a href="https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic2/2.4_inference_time_compute.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# LLM Engineering Essentials от Nebius Academy
Курс на github: [ссылка](https://github.com/Nebius-Academy/LLM-Engineering-Essentials/tree/main)
Курс сейчас находится в разработке, скоро появятся дополнительные материалы. [Подпишитесь, чтобы быть в курсе](https://academy.nebius.com/llm-engineering-essentials/update/)
# Р.2. Вычисление времени вывода

Теперь, когда мы понимаем, что успех рассуждений в математике можно частично объяснить простыми вычислениями, это вводит компромисс между вычислениями и качеством. Это поднимает важный вопрос: **При наличии определенного количества вычислений времени вывода на каждый вопрос, как мы можем его оптимально распределить?**
Если у нас достаточно вычислительных ресурсов только для одного запроса LLM, нам особо не о чем думать. Мы просто работаем и надеемся на лучшее. Но что, если мы сможем запросить LLM 10 раз по запросу одного пользователя? Или даже 1000 раз? Что, если мы столь же безрассудны, как OpenAI, которая [потратила более 10 тысяч долларов **за задачу**, когда они тестировали **o3** в призе ARC-AGI](https://arcprize.org/blog/oai-o3-pub-breakthrough).
В этом блокноте мы обсудим, как разумно использовать **вычисления времени вывода** посредством продуманной оркестровки.
# Параллельная и последовательная оркестровка
До недавних прорывов в области нелинейных рассуждений существовало несколько способов «раздуть» компьютер. Начнем с обсуждения двух самых простых вариантов (см. иллюстрацию ниже), а затем перейдем к более сложным.
- **Параллельно**: параллельное выполнение $N$ идентичных запросов с **ненулевой температурой**, а затем агрегирование результатов. Например, мы могли бы выбрать окончательный ответ, используя большинство голосов (эта стратегия довольно запутанно называется **Самосогласованность**). Более высокая температура гарантирует, что выходные данные будут достаточно различаться; это похоже на $N$ независимых экспертов, каждый из которых предлагает свое мнение. При наличии достаточного количества экспертов истина может быть раскрыта из множества ответов.
  В разделе практики мы реализовали для вас **самосогласованность**.
  Этот подход похож на **ансемблирование** в классическом машинном обучении: несколько достаточно разных моделей могут превзойти любую отдельную модель. Неудивительно, что самосогласованность довольно популярна благодаря своей мощности и простоте (по сравнению с другими подходами, которые мы обсудим дальше).
- **Последовательный**: заставить LLM неоднократно пересматривать свое решение «без присмотра», просто предлагая ему исправить свой предыдущий ответ. Сложная часть заключается в том, что [LLM часто не очень хороши в самокоррекции](https://arxiv.org/pdf/2310.01798). Итак, вам, вероятно, придется точно настроить LLM для этой задачи. Однако сначала вам нужно будет собрать набор данных, что является настоящей проблемой. Неудивительно, что мало кто пытается это сделать.
  <подробности>
  <summary>Как бы вы собирали данные для точной настройки LLM для самокоррекции? Нажмите здесь, если вам интересно.</summary>
  Поделюсь методом из [этой статьи](https://arxiv.org/pdf/2408.03314). Авторы:
  * Параллельная выборка 64 ответов при более высокой температуре.
  * Соединяем каждое правильное решение с четырьмя неправильными для создания многооборотных данных самокоррекции.
  * Использована метрика расстояния редактирования символов для определения приоритета выбора неправильных решений, тесно связанных с правильными.раствор. Это несколько наивный метод определения того, является ли одно решение редактированием другого, но он сработал!
  </подробнее>
* Наконец, если вы освоили оба подхода, вы можете объединить их в **гибридную стратегию**: самосогласованность в серии последовательностей переписывания!
<center>
<img src="https://drive.google.com/uc?export=view&id=17NfFNgMoWliod4rw3t9W4MGZBwYgiKbC" width=600 />
</center>
Источник изображения: [Оптимальное масштабирование вычислений во время тестирования LLM может быть более эффективным, чем масштабирование параметров модели] (https://arxiv.org/pdf/2408.03314) от Google DeepMind, это очень содержательная статья по теме вычислений во время вывода.
Но чтобы по-настоящему понять, что происходит на этом изображении, нам нужно сделать следующий шаг: **проверить решение, а не только ответ**.

# Модель вознаграждения процесса (PRM)
Раньше я много преподавал математику и твердо убежден, что важно проверять не только окончательный ответ, но и все решение. Даже совершенно неправильное решение иногда может привести к правильному ответу, как показано в следующем примере. Пытаясь упростить дробь \(\frac{15}{95}\), почему бы просто не сократить пятёрки как в числителе, так и в знаменателе? Разве это не даст правильный ответ?
$$\frac{1\color{red}{5}}{9\color{red}{5}} = \frac15$$
Давайте рассмотрим еще один пример с многошаговым решением:
**Задача**: Найти максимальный корень уравнения $\frac{x^3 - x^2}{x - 1} = 3x - 2$.
**Решение**:
1) Сначала разложим числитель в левой части на множители: $x^3 - x^2 = x^2(x - 1)$.
2) Теперь мы можем отменить члены $x - 1$, упростив уравнение до $x^2 = 3 - 2x$.
3) Сдвигая всё в одну сторону, получаем $x^2 - 3x + 2 = 0$.
4) Это уравнение имеет корни в точках $x = 1$ и $x = 2$.
5) Следовательно, максимальный корень равен $\boxed{2}$.
**Серьезной ошибкой в ​​решении** было сокращение $x - 1$ без учета того, что оно не может быть нулевым. $x = 1$ не может быть действительным корнем исходного уравнения, поскольку замена $x = 1$ приводит к появлению $\frac00$ в левой части. Однако окончательный ответ по-прежнему верен, потому что нам не нужно рассматривать $x = 1$ в этом контексте.
Обратите внимание, что и Лама-3.1-8B, и Лама-3.1-70B попадают в эту ловушку, тогда как Лама-3.1-405B правильно признает, что $x = 1$ не может быть решением. Не проверив решение, мы не поймем, в чем заключается проблема.
Предпринимались различные попытки оценить решения в целом (**Модели вознаграждения за результат**, **ORM**) или для каждого отдельного шага (**Модели вознаграждения за процесс**, **PRM**). ФРМ сложнее, интереснее и потенциально более эффективны, поэтому давайте обсудим их немного подробнее.

## Как определить шаги решения?
На этот вопрос нет простого ответа. В приведенном выше примере решение аккуратно разделено на этапы, но LLM не всегда это делает. Также может быть полезно разделить решения по `"\n\n"` (конец абзаца), но уравнения LaTeX, отображаемые на отдельных строках, могут вызвать проблемы с этим. Например, см. решение той же проблемы, предоставленное Llama-3.1-405B ниже; ничего хорошего не произойдет, если просто разбить по абзацам.
<подробности>
<summary>Нажмите, чтобы увидеть решение Ламы</summary>
Чтобы найти максимальный корень уравнения, нам сначала нужно его упростить.
Данное уравнение имеет вид $\frac{x^3 - x^2}{x - 1} = 3x - 2$.
Мы можем факторизовать числитель как $x^2(x - 1)$.
Итак, уравнение принимает вид $\frac{x^2(x - 1)}{x - 1} = 3x - 2$.
Теперь мы можем отменить члены $(x - 1)$:
$x^2 = 3x - 2$
Перестановка уравнения в квадратное уравнение:
$x^2 - 3x + 2 = 0$
Факторизация квадратного уравнения:
$(х - 2)(х - 1) = 0$
Решая $x$, мы получаем:
$x = 1$ или $x = 2$
Однако исходное уравнение не определено при $x = 1$, поэтому мы исключаем это решение.
Следовательно, максимальный корень уравнения равен $x = 2$.
</подробнее>
Итак, как правило, если вы хотите использовать PRM, лучше специально предложить вашему LLM что-то вроде `Split your solution into individual logical steps and keep each step in one line.`

## Настройка задачи
PRM оценивает не только отдельные шаги, но и **частичные решения**, например:
| Частичное решение | Значение |
|---------------|-------|
| **Шаг 1** | 0,98 |
| **Шаги 1+2** | 0,76 |
| **Шаги 1+2+3** | 0,21 |
| **Шаги 1+2+3+4** | 0,29 |
Это более логично, потому что отдельный шаг имеет смысл только с учетом того, что было раньше в решении.
Разумно рассматривать задачу моделирования вознаграждения процесса как двоичную классификацию («хорошо»/«плохо»), что естественным образом приводит к прогнозированию оценки от 0 до 1 («вероятности классов»). Как мы увидим, на самом деле в качестве целей мы можем использовать оценки вероятности классов.
LLM можно точно настроить на модель процессного вознаграждения, например, научив его отвечать на хорошие частичные решения с помощью токена `"+"` и на плохие с помощью `"-"`. (Или, в более общем смысле, предсказать `+` с вероятностью класса «хорошо» и `-` с вероятностью класса «плохо».)
В практической части мы попробуем LLM, настроенный на PRM.
**Примечание**. Конечно, LLM также можно использовать в качестве PRM без какой-либо тонкой настройки, в режиме **LLM как судья**. Для этого вам нужно будет предложить ему задуматься о потенциале каждого частичного решения и дать оценку, желательно по небольшой дискретной шкале (например, 1-5). Несмотря на заманчивость, этот подход не лишен оговорок:
* Мы возлагаем слишком много надежд на способность студентов-магистров рассуждать, полагая, что именно их способность рассуждать мы хотим улучшить или получить балл в первую очередь.
* Как правило, вы берете мощную модель в качестве судьи-судьи LLM, в то время как вы можете точно настроить меньшую модель, чтобы она стала достойным PRM. Если вам нужен оценщик только для однократной оценки чего-либо, это может быть нормально, но если вы хотите в дальнейшем использовать его для управления генерацией выводов, это может стать проблемой.

## Где взять данные для тренировок по ФРМ?
Это одна из тех ситуаций, когда получение данных является сложной задачей. Готового набора данных нет, а маркировка человеком будет ужасно дорогой. (Кроме того, созданные человеком этикетки для частичных растворов, скорее всего, будут плохо откалиброваны.)
Неудивительно, что более жизнеспособным подходом является неконтролируемый подход, основанный на запуске **развертывания по методу Монте-Карло** на каждом этапе решения. Это было предложено в статье [Math-Shepherd](https://arxiv.org/pdf/2312.08935) и работает следующим образом:
* Чтобы получить частичное решение, сгенерируйте большое количество $N$ его продолжений и проверьте, сколько из них приведут к правильному ответу. (К счастью, большинство наборов математических данных содержат правильные ответы.) Отношение допустимых продолжений к $N$ и будет оценкой. Кажется, это хорошая оценка «вероятности того, что это частичное решение хорошее».
<center>
<img src="https://drive.google.com/uc?export=view&id=1-FnTNw1GV9FMu1k4hKnNTjHZCjGlkq-1" width=600 />
</center>
Вы найдете код в практической части тетради.
**Слово предостережения**. Способность PRM обнаруживать математические и логические ошибки зависит от того, влияют ли эти ошибки в обучающих данных на ответы. Например, если набор обучающих данных содержит только описанную выше задачу, где отмена $x-1$ не влияет на ответ, PRM не узнает, что отмену следует выполнять ответственно. Однако, если набор обучающих данных PRM включает в себя задачу поиска **все** корней $\frac{x^3 - x^2}{x - 1} = 3x - 2$, это может научить PRM чему-то об отмене.

## Применения PRM и предостережения
Есть несколько способов использования PRM:
* В качестве награды за дальнейшую тонкую настройку LLM. Например, статья [Math-Shepherd](https://arxiv.org/pdf/2312.08935) демонстрирует, что RLHF с PRM, обученным математическим задачам, может улучшить математические способности LLM.
* В качестве альтернативы основному голосованию в режиме **Самосогласованность**. Действительно, основное голосование выбирает только самый популярный ответ, но не учитывает качество решения. Выбор решения с максимальным показателем PRM может привести к получению более правильных результатов. Именно это было использовано в [документе DeepMind](https://arxiv.org/pdf/2408.03314) о котором мы упоминали ранее и из которого позаимствовали уже знакомую картину.
  <center>
  <img src="https://drive.google.com/uc?export=view&id=17NfFNgMoWliod4rw3t9W4MGZBwYgiKbC" width=600 />
  </center>
  Существует несколько способов оценки всего решения с помощью PRM.
  * Оцените каждое частичное решение (шаг 1, шаги 1+2, шаги 1+2+3,...), а затем суммируйте баллы.
  * Просто оцените все решение, не беспокоясь о его префиксах.
  Есть свидетельства того, что второй подход работает достаточно хорошо. Однако стоит отметить, что PRM, которые обучены оценивать частичные решения, более точны, чем ORM (модели вознаграждения за результат), которые обучены оценивать только полные решения.
* В настройках последовательной перезаписи вместо выбора окончательной перезаписи мы можем использовать PRM для оценки каждого решения в цепочке и выбора лучшего решения.
* Наконец, мы можем использовать PRM на промежуточных этапах для управления генерацией. И об этом мы поговорим дальше!
Хотя это и круто, у PRM есть свои проблемы. Они не совсем надежны и, что еще хуже, плохо переносятся между разными моделями. Вам определенно следует быть осторожным при оценке решений DeepSeek R1 с использованием PRM, обученного на решениях, созданных Mistral.

# ORM как PRM
В некоторых ситуациях у нас есть **ORM** (**Модель вознаграждения за результат**), которая оценивает только полные решения вместо PRM. Например, при генерации кода роль ORM может выполнять набор тестов или других автоматически проверяемых требований.
В таких случаях вы можете использовать **просмотр** для оценки частичных решений. Идея для вас не нова:
* Для данного частичного решения создайте несколько полных продолжений (**развертываний**).
* Оцените каждое из продолжений и усредните их балл. Это даст вам оценку частичного решения.
**Примечание**. Упреждающий поиск хорош не только для моделирования модели вознаграждения процесса с моделью вознаграждения за результат. Даже если у вас есть PRM, иногда полезно получить частичное решение,
- генерирование на несколько шагов вперед,
- а затем забить это более длинное частичное решение.
Более того, вы можете использовать несколько предварительных просмотров; потенциально это может дать вам более надежное вознаграждение.
**Примечание**. Иногда ваш ORM — это просто детерминированная проверка. Например, в задаче кодирования, где автоматические тесты могут быть запущены после получения полного решения. Конечно, в этом случае упреждающий поиск — отличный способ найти частичные решения.

# Генерация и оркестровка нелинейных рассуждений на основе PRM
Как мы уже упоминали ранее, люди решают проблемы нелинейным способом: исследуют разные пути решения, отбрасывая одни и уделяя больше внимания другим. Самосогласованность — это грубая аналогия этого процесса — исследования множества независимых траекторий мышления, — но довольно грубая.
Прежде чем студенты LLM научились выполнять нелинейные рассуждения самостоятельно, появились различные подходы к их организации. В этом разделе мы обсудим некоторые из них.

##Древо мыслей и лучевой поиск
Три мысли, впервые представленные в [одноименной статье](https://arxiv.org/pdf/2305.10601) — это довольно простая реализация нелинейной генерации. Грубая идея состоит в том, чтобы исследовать дерево потенциальных решений, где каждая вершина является «мыслью» (шагом решения), с помощью поиска в ширину (BFS) или поиска в глубину (DFS).
<center>
<img src="https://drive.google.com/uc?export=view&id=16J6w1QdzkX81zm10H1hKetdwvhHKOlz_" width=600 />
[Источник](https://arxiv.org/pdf/2305.10601)
</center>
Чтобы алгоритм Тройки мыслей сформировался, нам необходимо выбрать:
* Потенциальная **деревовидная структура**: **максимальное количество ветвей** в каждой вершине, **максимальная глубина**.
* **Как мы пробуем следующие «мысли»**. По сути, мы либо делаем несколько параллельных запросов к LLM, чтобы «сгенерировать следующий логический шаг», либо просим его «сгенерировать несколько вариантов следующего логического шага» в одном приглашении.
  Последний подход может быть полезен для предотвращения дублирования, когда вариантов не слишком много (что-то вроде поиска кратчайшего маршрута в графе). В противном случае я бы выбрал параллельные запросы.
* Как мы решаем, какие пути исследовать дальше, а какие отказаться. Для этого вам нужен способ **оценки** вершины (состояния) или частичного решения, которое к ней привело. В оригинальной статье «Древо мыслей» использовалась модель LLM в качестве судьи (что было хорошо мотивировано конкретными задачами, которые они рассматривали), но для большинства задач разумным выбором была бы обученная **Модель процесса вознаграждения**.
  Конкретные критерии, основанные на PRM, могут использоваться для определения того, какие отрасли являются безнадежными, и для определения приоритетности высокопотенциальных отраслей.
  Особым случаем Дерева мыслей является **Поиск по лучу**, вариант поиска в ширину, который сохраняет не более $B$ (**размер луча**) вершин на каждом уровне. Если $B=2$, это работает следующим образом:
  * Для начала генерируются 2 варианта первого шага.
  * Во время каждого из следующих взаимодействий генерируются два следующих шага для каждого из двух имеющихся у нас промежуточных решений.
  * 2 из них, набравшие наибольшее количество очков, передаются на следующую итерацию.
  В некоторых вариантах изначально генерируются 4 первых шага, а 2 из них с наибольшим количеством очков отбираются для дальнейшего создания.
<center>
<img src="https://drive.google.com/uc?export=view&id=1y3io1RqfqyIKcfu7kxRIiZKNKGMXtt9P" width=500 />
</center>
В практической части мы реализуем поиск по лучу.

## За Древом мыслей
В 2023 и 2024 годах появились еще более сложные варианты оркестровки, такие как [График мыслей](https://arxiv.org/pdf/2308.09687) или [Алгоритм мыслей](https://arxiv.org/pdf/2308.10379). Еще одним интересным примером управляемой стратегии генерации на основе дерева является **Поиск по дереву Монте-Карло** (**MCTS**), который мы обсудим в следующей записной книжке. Однако эти подходы являются дорогостоящими и сложными, и из-за этого с точки зрения, они имеют тенденцию отставать от простого использования более крупных LLM или LLM с собственными нелинейными рассуждениями, такими как o1 OpenAI и DeepSeek R1. Тем не менее, интересно проверить эти подходы и поразмыслить, как они резонируют с тем, что происходит в R1 и подобных моделях.
<center>
<img src="https://drive.google.com/uc?export=view&id=1WZWjI7aY3Vu0zEsAO8u7R73iwsC6KJeq" width=600 />
[Источник](https://arxiv.org/pdf/2308.09687)
</center>

# Практические рекомендации и законы масштабирования выводов
Как же при таком богатом выборе подходов выбрать лучший? Для начала нам нужно понять наш **бюджет вывода**, то есть сумму денег (или, проще говоря, вызовы вывода), которые мы можем потратить на обработку одного запроса. Имея в виду бюджет вывода, мы можем попробовать выбрать
* **LLM, который мы хотим использовать**: какой уровень размера, возможности нелинейного рассуждения и т. д.
* **Как расширить бюджет вывода**: такие методы, как самосогласованность, поиск лучей и многое другое.
  Например, если наш бюджет позволяет $N$ вызовов LLM на каждый запрос, мы можем повысить самосогласованность с помощью $N$ параллельных вызовов.
  <подробности>
  <summary>Если вам интересно, посмотрите очень грубое сравнение поиска луча и самосогласованности.</summary>
  Предположим, что размер луча равен $B$, каждое решение состоит из $D$ мыслей по $T$ токенов в каждой, а задача состоит из $P$ токенов. Кроме того, будет разумно подсчитать, что обработка входного токена стоит $\$c$, а генерация одного выходного токена стоит $\$3c$ (это более или менее справедливо для большинства API). Теперь с поиском луча:
  * Первый шаг обойдется нам в $B\cdot(cP + 3cT)$,
  * Второй шаг будет стоить $B\cdot([cP + cT] + 3cT)$ (промпты теперь проблема + первая мысль),
  * Третий шаг будет стоить $B\cdot([cP + 2cT] + 3cT)$ (теперь промпты: проблема + первая мысль + вторая мысль),
  * ...
  * Последний, $D$-й шаг будет стоить $B\cdot([cP + (D-1)\cdot cT] + 3cT)$
  В целом это дает
  $$Bc\cdot\left(DP + T + 2T + \ldots + (D-1)T + 3T \right)=
  Bc\cdot\left(DP + \left[\frac12D(D-1) + 3\right]T \right)$$
  Для самосогласованности параллельных запросов $B$ нам нужны вызовы $B$ с входными токенами $P$ и выходными токенами $DT$, что приводит к стоимости
  $$B\cdot(cP + 3cDT) = Bc\cdot(P + 3DT)$$
  Обратите внимание: здесь мы не учитывали вызовы $BD$ PRM!
  Обычно поиск луча обходится дороже, чем поиск самосогласованности, но теперь мы также видим, что из-за того, что стоимость поиска луча квадратична по $D$ (длина решения в «мыслях»), для длинных решений поиск луча будет намного дороже.
  </подробнее>
Некоторые ключевые соображения:
* **Большие модели и более разумные стратегии**
  Обновление до более мощного LLM (например, **Llama-3.1-405B** вместо **Llama-3.1-8B**) может обеспечить значительное повышение качества, но за очень высокую цену. В некоторых случаях меньший LLM в сочетании с самосогласованностью или более продвинутой стратегией может превзойти более крупную модель при меньших затратах.
  Однако с ростом бюджета времени вывода более крупный LLM, запрашиваемый напрямую, может в конечном итоге стать более предпочтительным выбором, чем сложная, подверженная ошибкам оркестровка.
  Мы рассмотрим этот компромисс в практическом разделе.
* **Компромиссы при генерации на основе PRM**
  Древовидный ген, управляемый PRMрацион является дорогостоящим и более сложным в настройке. (И хорошие ФРМ не валяются просто так!) Если вы ищете отправную точку, самосогласованность — надежный выбор.
  В документе [Оптимальное масштабирование вычислений во время тестирования LLM может быть более эффективным, чем масштабирование параметров модели](https://arxiv.org/pdf/2408.03314) в документе представлены несколько экспериментов, направленных на выявление **законов масштабирования вывода** — закономерностей того, как эффективность развивается по мере изменения бюджетов вывода.
  Результаты ниже относятся к [тесту MATH](https://huggingface.co/datasets/nlile/hendrycks-MATH-benchmark).
  * При меньших затратах на вывод лучевой поиск неизменно превосходит другие подходы (при условии, что для его выполнения достаточно вычислительных ресурсов).
  * Для больших бюджетов вывода используется другой метод: параллельная генерация нескольких решений и выбор лучшего на основе максимального показателя PRM.
  Конечно, нет никакой гарантии, что эти результаты будут распространяться на все LLM и наборы данных. Однако они настоятельно предполагают, что даже при большом бюджете вывода вам не обязательно нужны очень сложные стратегии для оптимизации производительности.
  <center>
  <img src="https://drive.google.com/uc?export=view&id=1Bh81Ycx-2H0676F7MTpnRhGYFMg8QMyo" width=600 />
  </center>

# Готовы к большему?
Этот блокнот является частью более крупного бесплатного курса — **LLM Engineering Essentials** — где вы пойдете еще дальше в своем обучении и создадите сервис для создания умных, похожих на людей NPC.
🎓Скоро появятся новые материалы. Нажмите на ссылку ниже, чтобы подписаться на обновления и убедиться, что вы ничего не пропустите:
[Будьте в курсе](https://academy.nebius.com/llm-engineering-essentials/update/)

# Практическое занятие
У нас амбициозные планы на эту практику. Мы будем:
* Практикуйте балансирование бюджета вывода между использованием крупных LLM и использованием сложных стратегий оркестрации с меньшими LLM.
* Реализуйте лучевой поиск, используя PRM ​​как на основе модели, так и на основе достоверности.
Если вы столкнулись с какими-либо трудностями или просто хотите увидеть наши решения, загляните в [Блокнот решений](https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic2/r.2_inference_time_compute.ipynb).

## Готовимся

In [1]:
!pip install -q openai

In [2]:
import os

with open("nebius_api_key", "r") as file:
    nebius_api_key = file.read().strip()

os.environ["NEBIUS_API_KEY"] = nebius_api_key

В этом блокноте мы будем довольно часто вызывать API, поэтому давайте определим функцию быстрого доступа, чтобы избежать повторения всего кода:

In [3]:
from openai import OpenAI

nebius_client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

llama_8b_model = "meta-llama/Meta-Llama-3.1-8B-Instruct"

def prettify_string(text, max_line_length=80):
    """Prints a string with line breaks at spaces to prevent horizontal scrolling.

    Args:
        text: The string to print.
        max_line_length: The maximum length of each line.
    """

    output_lines = []
    lines = text.split("\n")
    for line in lines:
        current_line = ""
        words = line.split()
        for word in words:
            if len(current_line) + len(word) + 1 <= max_line_length:
                current_line += word + " "
            else:
                output_lines.append(current_line.strip())
                current_line = word + " "
        output_lines.append(current_line.strip())  # Append the last line
    return "\n".join(output_lines)

def answer_with_llm(prompt: str,
                    system_prompt="You are a helpful assistant",
                    max_tokens=512,
                    client=nebius_client,
                    model=llama_8b_model,
                    prettify=True,
                    temperature=0.7) -> str:

    messages = []

    if system_prompt:
        messages.append(
            {
                "role": "system",
                "content": system_prompt
            }
        )

    messages.append(
        {
            "role": "user",
            "content": prompt
        }
    )

    completion = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature
    )

    if prettify:
        return prettify_string(completion.choices[0].message.content)
    else:
        return completion.choices[0].message.content

# Практика, часть 1. Более крупные модели против более разумных стратегий: классификация ролей в диалоге с помощью CoT и самосогласованности
В этой задаче мы будем работать с [тестом DialogRE](https://github.com/nlpdata/dialogre) и классифицировать роли в диалогах с помощью моделей Llama и Nebius AI Studio.
Начнем с загрузки набора данных.

In [4]:
!wget https://raw.githubusercontent.com/nlpdata/dialogre/master/data/dev.json -O dialogre_dev.json

--2025-06-04 09:07:43--  https://raw.githubusercontent.com/nlpdata/dialogre/master/data/dev.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 754825 (737K) [text/plain]
Saving to: ‘dialogre_dev.json’

dialogre_dev.json   100%[===================>] 737.13K  --.-KB/s    in 0.02s   

2025-06-04 09:07:44 (31.6 MB/s) - ‘dialogre_dev.json’ saved [754825/754825]



In [5]:
import json

with open('dialogre_dev.json', 'r') as f:
    dialog_data_raw = json.load(f)

Важно смотреть на данные.

In [6]:
len(dialog_data_raw)

358

In [7]:
dialog_data_raw[0]

[['Speaker 1: Hey!',
  'Speaker 2: Hey.',
  "Speaker 3: Hey, man. What's up?",
  "Speaker 1: Maybe you can tell me. My agent would like to know why I didn't show up at the audition I didn't know I had today. The first good thing she gets me in weeks. How could you not give me the message?!",
  "Speaker 3: Well, I'll tell ya I do enjoy guilt, but, ah, it wasn't me.",
  'Speaker 2: Yes, it was! It was him! Uh huh! Okay, it was me!',
  'Speaker 1: How is it you?',
  "Speaker 2: Well, it was just, it was all so crazy, you know. I mean, Chandler was in the closet, counting to 10, and he was up to 7 and I hadn't found a place to hide yet. I-I-I meant to tell you, and I wrote it all down on my hand. See, all of it.",
  "Speaker 1: Yep, that's my audition.",
  'Speaker 4: See, now this is why I keep notepads everywhere.',
  "Speaker 2: Yep, and that's why we don't invite you to play.",
  'Speaker 5: What is the great tragedy here? You go get yourself another appointment.',
  'Speaker 1: Well, 

Как видите, для каждого диалога помечено множество разных ролей персонажей.
Вот список всех ролей:

In [8]:
all_relations = []
for dialog in dialog_data_raw:
    for relation in dialog[1]:
        for individual_relation in relation['r']:
            if not individual_relation in all_relations:
                all_relations.append(individual_relation)
all_relations

['per:title',
 'per:alternate_names',
 'per:client',
 'per:friends',
 'unanswerable',
 'per:spouse',
 'per:children',
 'per:parents',
 'per:age',
 'per:siblings',
 'per:roommate',
 'per:negative_impression',
 'per:pet',
 'per:positive_impression',
 'per:girl/boyfriend',
 'org:employees_or_members',
 'per:employee_or_member_of',
 'per:dates',
 'per:boss',
 'per:subordinate',
 'per:other_family',
 'org:students',
 'per:major',
 'per:schools_attended',
 'per:origin',
 'gpe:visitors_of_place',
 'per:visited_place',
 'per:alumni',
 'per:works',
 'per:place_of_residence',
 'gpe:residents_of_place',
 'per:place_of_work',
 'per:date_of_birth',
 'per:acquaintance',
 'per:neighbor',
 'gpe:births_in_place',
 'per:place_of_birth']

Мы не хотим слишком усложнять задачу, поэтому будем придерживаться более простых и частых ролей. Мы также исключим `per:alternate_names`, поскольку он слишком избыточен в наборе данных и слишком сильно повлияет на балансировку классов.

In [9]:
our_relations = [
    'per:friends', 'per:spouse', 'per:children', 'per:parents',
    'per:siblings', 'per:girl/boyfriend', 'per:boss', 'per:subordinate'
]

Мы выберем только те диалоги, которые содержат хотя бы один из выбранных статусов отношений.

In [10]:
dialog_data = []
for dialog in dialog_data_raw:
    current_dialog = dialog[0]
    current_relations = []
    for relation in dialog[1]:
        if relation['r'][0] in our_relations:
            current_relations.append({
                'x': relation['x'],
                'y': relation['y'],
                'r': relation['r'][0]
            })
    if len(current_relations) > 0:
        dialog_data.append([current_dialog, current_relations])

In [11]:
len(dialog_data)

180

In [12]:
dialog_data[1]

[['Speaker 1, Speaker 2: Hi',
  'Speaker 3: Hi! Hey mom.',
  'Speaker 4: This is such a great party! 35 years. Very impressive, do you guys have any pearls of wisdom?',
  'Speaker 2: Jack?',
  'Speaker 1: Why would you serve food on such a sharp stick?',
  'Speaker 3: That’s a good question, dad. That’s a good question…',
  'Speaker 4: Hmmm….'],
 [{'x': 'Speaker 1', 'y': 'Speaker 2', 'r': 'per:spouse'},
  {'x': 'Speaker 1', 'y': 'Speaker 3', 'r': 'per:children'},
  {'x': 'Speaker 2', 'y': 'Speaker 1', 'r': 'per:spouse'},
  {'x': 'Speaker 2', 'y': 'Speaker 3', 'r': 'per:children'},
  {'x': 'Jack', 'y': 'Speaker 3', 'r': 'per:children'},
  {'x': 'Speaker 3', 'y': 'Speaker 2', 'r': 'per:parents'},
  {'x': 'Speaker 3', 'y': 'Speaker 1', 'r': 'per:parents'},
  {'x': 'Speaker 3', 'y': 'Jack', 'r': 'per:parents'}]]

Чтобы еще больше упростить задачу, для каждого диалога мы выберем только одну роль для прогнозирования. Мы будем использовать случайную выборку, но **исправим случайное начальное число**, чтобы процедура отбора была воспроизводимой. А для экономии времени и денег мы возьмем только 50 первых диалогов из набора `dev`.

In [13]:
import numpy as np

np.random.seed(28)
dialog_data_short = [[dialog, np.random.choice(relations)] for dialog, relations in dialog_data[:50]]

In [14]:
dialog_data_short[1]

[['Speaker 1, Speaker 2: Hi',
  'Speaker 3: Hi! Hey mom.',
  'Speaker 4: This is such a great party! 35 years. Very impressive, do you guys have any pearls of wisdom?',
  'Speaker 2: Jack?',
  'Speaker 1: Why would you serve food on such a sharp stick?',
  'Speaker 3: That’s a good question, dad. That’s a good question…',
  'Speaker 4: Hmmm….'],
 {'x': 'Speaker 1', 'y': 'Speaker 3', 'r': 'per:children'}]

In [15]:
verdicts_true = [relations['r'] for _, relations in dialog_data_short]

В задачах машинного обучения всегда важно следить за распределением целевых меток. В нашем случае это не сбалансировано: друзей и девушек/парней гораздо больше, чем других ролей. На данный момент мы не будем принимать это во внимание, но в реальной задаче Data Science мы попытаемся скорректировать наши метрики, чтобы принять во внимание дисбаланс классов.

In [16]:
from collections import Counter

relations_counter = Counter(verdicts_true)

relations_counter

Counter({'per:friends': 11,
         'per:children': 3,
         'per:parents': 4,
         'per:siblings': 6,
         'per:spouse': 3,
         'per:girl/boyfriend': 16,
         'per:boss': 5,
         'per:subordinate': 2})

## Вариант 1: Большая модель с простой стратегией CoT
Для начала мы будем использовать Llama-3.1-405B с простой стратегией анализа запрограммированных ответов CoT +.
Мы могли бы использовать цепочку LLM со второй моделью для извлечения ответа, но просто проанализировать ее не составит труда. Единственная нетривиальная вещь, которую мы вводим на этапе синтаксического анализа, связана с наблюдением, что иногда LLM предсказывают `boyfriend` или `girlfriend` вместо `girl/boyfriend`. Итак, мы просто вручную сопоставляем `boyfriend` или `girlfriend` с `girl/boyfriend`.

In [17]:
import re

class RelationClassifier():
    def __init__(self, client, model):
        self.client = client
        self.model = model
        self.raw_classes = ["friends", "spouse", "children", "parents",
                            "siblings", "girl/boyfriend", "boss", "subordinate"]

    def predict(self, dialog, character_x, character_y, verbose=False):
        reasoning_completion = self.client.chat.completions.create(
            messages=[
                {
            "role": "user",
            "content": f"""You are an expert in Natural Nanguage Understanding.
You are gived a dialog and two characters participating or mentioned in this dialog.
You need to predict relationships between these characters choosing from the following list:
- friends
- spouse
- children
- parents
- siblings
- girl/boyfriend
- boss
- subordinate
Provide a clear reasoning justifying your choice.
Then write your final answer after #VERDICT:
Your final answer should be exactly one of the relationship types from the list, with no explanation.
Now, take a deep breath and work out this problem step by step.

DIALOG: {dialog}

FIRST CHARACTER: {character_x}

SECOND CHARACTER: {character_y}

REASONING:"""
                }
            ],
            model=self.model,
            temperature=0.1
            )
        reasoning = reasoning_completion.choices[0].message.content

        # Extract whatever is after #VERDICT:
        re_match = re.search(r"#VERDICT(.*)", reasoning, re.DOTALL)
        if re_match:
            extracted_answer = re_match.group(1).strip().lower()
        else:
            extracted_answer = "Failed to parse"

        # Parse the answer
        verdict = extracted_answer.lower().strip("'\".:; ")
        if verdict == "boyfriend" or verdict == "girlfriend":
            verdict = "girl/boyfriend"
        if verdict in self.raw_classes:
            verdict = "per:" + verdict
        else:
            verdict = "per:failed"

        if verbose:
            return {
                "reasoning_completion": reasoning_completion,
                "extracted_answer": extracted_answer,
                "verdict": verdict
            }
        else:
            return verdict

In [18]:
client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

classifier_llama_405b = RelationClassifier(
    client=client, model="meta-llama/Meta-Llama-3.1-405B-Instruct"
    )

Хорошо бы начать записывать результаты

In [19]:
completions_log = dict() # Raw completions
verdicts_log = dict() # Final verdicts

In [20]:
# The tqdm library allows to create progress bars for cycles
from tqdm import tqdm

current_configuration = "Meta-Llama-3.1-405B-Instruct, no enhancements"
results = []

# If you're short in compute, try for dialog_data_short[:10]
for dialog, relations in tqdm(dialog_data_short):
    results.append(classifier_llama_405b.predict(dialog, relations['x'], relations['y'], verbose=True))

completions_log[current_configuration] = results

100%|██████████| 50/50 [05:07<00:00,  6.15s/it]


Давайте посмотрим на результаты:

In [21]:
results[1]

{'reasoning_completion': ChatCompletion(id='chatcmpl-db1f4b50fa8743d49bc414ed1a27693b', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='To determine the relationship between Speaker 1 and Speaker 3, we need to analyze the dialog for clues.\n\n1. Speaker 3 greets with "Hi! Hey mom." However, this greeting is not directed at Speaker 1, as Speaker 3 later addresses Speaker 1 as "dad." This indicates a familial relationship.\n\n2. Speaker 3\'s statement "That’s a good question, dad. That’s a good question…" directly addresses Speaker 1 as "dad," which is a clear indicator of their relationship.\n\nGiven these points, it is clear that Speaker 1 and Speaker 3 are related as parent and child.\n\n#VERDICT: parents', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=None), stop_reason=None)], created=1749028078, model='meta-llama/Meta-Llama-3.1-405B-Instruct', object='chat.compl

Для начала проверим процент случаев, когда нам не удалось разобрать вердикт:

In [22]:
verdicts = [result["verdict"] for result in results]
print(sum([verdict == "per:failed" for verdict in verdicts]) / len(verdicts))

0.0


Теперь давайте проверим точность наших прогнозов.

In [23]:
import numpy as np
def accuracy_score(y_true, y_pred):
    return sum(np.array(y_true) == np.array(y_pred)) / len(y_true)

accuracy_score(verdicts_true, verdicts)

np.float64(0.76)

Для начала это неплохо!

## Вариант 2: Меньший LLM
Посмотрим, как с этой задачей справится модель меньшего размера **Лама-3.1-8Б**!

In [24]:
classifier_llama_8b = RelationClassifier(
    client=client, model="meta-llama/Meta-Llama-3.1-8B-Instruct"
    )

current_configuration = "Meta-Llama-3.1-8B-Instruct, no enhancement"
results = []
# do it for patient_visits[-10:] to save time
for dialog, relations in tqdm(dialog_data_short):
    results.append(classifier_llama_8b.predict(dialog, relations['x'], relations['y'], verbose=True))

completions_log[current_configuration] = results

100%|██████████| 50/50 [04:23<00:00,  5.26s/it]


In [25]:
results[-3]

{'reasoning_completion': ChatCompletion(id='chatcmpl-59e3bf9003514fd0979cd9a2654fb84c', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Based on the dialog, I can infer the relationship between Mr. Kostelick and Speaker 2 as follows:\n\n* Mr. Kostelick mentions that Speaker 2 needs to stop by his office at the end of the day, which suggests that Mr. Kostelick is in a position of authority over Speaker 2.\n* Speaker 2 is defensive and nervous when speaking to Mr. Kostelick, which further suggests a power imbalance in their relationship.\n* Speaker 2 also mentions that Mr. Kostelick wants to make them a processing supervisor, which implies that Mr. Kostelick has the power to promote or demote Speaker 2.\n\nGiven these clues, I believe that Mr. Kostelick is likely Speaker 2's boss.\n\n#VERDICT\nboss", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=None), stop_reason=Non

Посмотрим, сколько вердиктов разобрано неправильно:

In [26]:
verdicts = [result["verdict"] for result in results]
print(sum([verdict == "per:failed" for verdict in verdicts]) / len(verdicts))

0.2


И, наконец, точность:

In [27]:
accuracy_score(verdicts_true, verdicts)

np.float64(0.54)

Это еще хуже.

## Вариант 3: Меньший LLM с самосогласованностью

Поколения LLM будут меняться от итерации к итерации, и ответы тоже могут меняться. И на самом деле точность приведенного выше алгоритма нестабильна.

Мы воспользуемся этим как еще одной возможностью улучшения качества. Есть основания предполагать, что даже если одно рассуждение может быть ложным, несколько попыток LLM рассуждения могут раскрыть истину.
Самый популярный подход называется **Самосогласованность**. Это работает следующим образом:
- Сгенерировать несколько (скажем, 5 или 7) путей рассуждения, извлечь ответ из каждого из них,
- Выберите наиболее частый вариант.
Это похоже на большинство голосов нескольких одинаковых LLM.

In [28]:
dialog, relations = dialog_data_short[5]
for _ in range(5):
    print(classifier_llama_8b.predict(
        dialog, relations['x'], relations['y'], verbose=True
        )["reasoning_completion"].choices[0].message.content)
    print("\n###\n")

Based on the dialog, I can infer the relationship between Speaker 2 and Speaker 3.

Speaker 2 mentions "my husband" when referring to Speaker 3, which indicates that Speaker 3 is Speaker 2's spouse.

#VERDICT
spouse

###

Based on the dialog, I can infer the relationship between Speaker 2 and Speaker 3.

Speaker 2 mentions "my husband" when referring to Speaker 3, which implies that Speaker 3 is Speaker 2's spouse.

#VERDICT
spouse

###

Based on the dialog, I can infer the relationship between Speaker 2 and Speaker 3.

Speaker 2 mentions "my husband" when referring to Speaker 3, which indicates that Speaker 3 is Speaker 2's spouse.

#VERDICT
spouse

###

Based on the dialog, I can infer the relationship between Speaker 2 and Speaker 3 as follows:

Speaker 2 mentions "my husband" when referring to Speaker 3, which indicates that Speaker 3 is Speaker 2's spouse.

#VERDICT
spouse

###

Based on the dialog, I can infer the relationship between Speaker 2 and Speaker 3 as follows:

Speaker 

Теперь давайте создадим классификатор на основе самосогласованности.

In [37]:
from collections import Counter

def most_frequent(List):
    occurence_count = Counter(List)
    return occurence_count.most_common(1)[0][0]

class RelationClassifierSelfConsistency():
    def __init__(self, client: OpenAI, model: str, n_trials: int = 5):
        self.client = client
        self.model = model
        self.n_trials = n_trials
        self.raw_classes = ["friends", "spouse", "children", "parents",
                            "siblings", "girl/boyfriend", "boss", "subordinate"]

    def predict(self, dialog, character_x, character_y, verbose=False):
        reasoning_completion = self.client.chat.completions.create(
            messages=[
                {
            "role": "user",
            "content": f"""You are an expert in Natural Nanguage Understanding.
You are gived a dialog and two characters participating or mentioned in this dialog.
You need to predict relationships between these characters choosing from the following list:
- friends
- spouse
- children
- parents
- siblings
- girl/boyfriend
- boss
- subordinate
Provide a clear reasoning justifying your choice.
Then write your final answer after #VERDICT:
Your final answer should be exactly one of the relationship types from the list, with no explanation.
Now, take a deep breath and work out this problem step by step.

DIALOG: {dialog}

FIRST CHARACTER: {character_x}

SECOND CHARACTER: {character_y}

REASONING:"""
                }
            ],
            model=self.model,
            temperature=0.7,
            n=self.n_trials # That's the main difference.
            )

        verdicts = []
        for i in range(self.n_trials):
            reasoning = reasoning_completion.choices[i].message.content

            # Extract whatever is after #VERDICT:
            re_match = re.search(r"#VERDICT(.*)", reasoning, re.DOTALL)
            if re_match:
                extracted_answer = re_match.group(1).strip()
            else:
                extracted_answer = "Failed to parse"

            # Parse the answer
            verdict = extracted_answer.lower().strip("'\".:; ")
            if verdict == "boyfriend" or verdict == "girlfriend":
                verdict = "girl/boyfriend"
            if verdict in self.raw_classes:
                verdict = "per:" + verdict
            else:
                verdict = "per:failed"

            verdicts.append(verdict)

        final_verdict = most_frequent(verdicts)
        if verbose:
            return {
                "reasoning_completion": reasoning_completion,
                "verdicts": verdicts,
                "verdict": final_verdict
            }
        else:
            return final_verdict

Теперь давайте посмотрим, способна ли самосогласованность улучшить результаты **Llama-3.1-8B**.
**Это может занять время!**

In [38]:
classifier_sc = RelationClassifierSelfConsistency(
    client=client, model="meta-llama/Meta-Llama-3.1-8B-Instruct", n_trials=5
)

Давайте классифицируем все утверждения и проверим точность.

In [39]:
from tqdm import tqdm

current_configuration = "Meta-Llama-3.1-8B-Instruct, self-consistency"
results = []
# If you want, do it for dialog_data_short[-20:] to save time
for dialog, relations in tqdm(dialog_data_short):
    results.append(classifier_sc.predict(dialog, relations['x'], relations['y'], verbose=True))

completions_log[current_configuration] = results

100%|██████████| 50/50 [03:08<00:00,  3.77s/it]


Как и раньше, мы проверяем как степень неудачного извлечения, так и точность классификации.

In [40]:
verdicts = [result["verdict"] for result in results]
print(sum([verdict == "per:failed" for verdict in verdicts]) / len(verdicts))

0.18


In [41]:
verdicts_log[current_configuration] = verdicts

In [42]:
accuracy_score(verdicts_true[10:], verdicts[10:])

np.float64(0.65)

Примерно так же, как у нас было с Ламой-3.1-405b. Теперь давайте сохраним наши логи, чтобы не потерять все результаты:

In [35]:
import pickle

pickle.dump(completions_log, open("completions_log.pkl", "wb"))
pickle.dump(verdicts_log, open("verdicts_log.pkl", "wb"))

## Расчет стоимости
Чтобы завершить задачу классификатора диалоговых отношений, давайте рассчитаем стоимость LLM API для каждого из сценариев, используя
* Цены взяты из [ссылки на модели Nebius AI Studio](https://studio.nebius.ai/)
* Токены учитываются по промптам и завершениям, которые мы старательно регистрировали.

In [43]:
# Models costs, per 1M tokens:
costs = {
    '405B': {
        'input': 1,
        'output': 3
    },
    '8B': {
        'input': 0.02,
        'output': 0.06
    }
}

for key, value in completions_log.items():
    print(f'=== With {key} ===')
    total_input_tokens = 0
    total_output_tokens = 0
    for result in value:
        for k, v in result.items():
            if 'completion' in k:
                if isinstance(v, list):
                    for completion in v:
                        total_input_tokens += completion.usage.prompt_tokens
                        total_output_tokens += completion.usage.completion_tokens
                else:
                    total_input_tokens += v.usage.prompt_tokens
                    total_output_tokens += v.usage.completion_tokens
    if '405' in key:
        model_size = '405B'
    elif '8B' in key:
        model_size = '8B'
    else:
        print('And what is that?..')
    input_cost = total_input_tokens / 1000000 * costs[model_size]['input']
    output_cost = total_output_tokens / 1000000 * costs[model_size]['output']
    total_cost = input_cost + output_cost

    print(f'''
        Input cost: {input_cost}
        Output cost: {output_cost}
        Total cost: {total_cost}
              ''')

=== With Meta-Llama-3.1-405B-Instruct, no enhancements ===

        Input cost: 0.024059
        Output cost: 0.029454
        Total cost: 0.053513000000000005
              
=== With Meta-Llama-3.1-8B-Instruct, no enhancement ===

        Input cost: 0.00045618
        Output cost: 0.00047453999999999997
        Total cost: 0.0009307199999999999
              
=== With Meta-Llama-3.1-8B-Instruct, self-consistency ===

        Input cost: 0.00045618
        Output cost: 0.00262992
        Total cost: 0.0030861
              


Как мы видим, даже при самосогласованности Лама-3.1-8Б всё равно на порядок дешевле Ламы-3.1-405Б. Более того, мы могли бы значительно увеличить `n_trials` до того, как достигнем цены на более крупную модель.
**Ключевой вывод**: всегда помните о компромиссе между более широкой моделью и более разумной стратегией и принимайте во внимание затраты.

# Практика, часть 2. Теперь ваша очередь!
Теперь пришло ваше время использовать `MMLUEvaluator` с пользой, с которой мы играли в предыдущем блокноте! Выберите одно из полей, связанных с математикой (вы можете проверить выбор в [статье](https://arxiv.org/pdf/2009.03300)) и выбрать первые 50 примеров.
**Ваша задача.** Сравните точность этого небольшого набора данных при использовании:
- **Лама-3.1-70Б** с подавлением ЦТ,
- **Лама-3.1-8Б** с подавлением ЦТ,
- **Лама-3.1-70Б** с базовым ЦТ,
- **Лама-3.1-8Б** с базовым ЦТ,
- **Лама-3.1-8Б** с самосогласованием.
Также сравните стоимость обработки 50 примеров с:
- **Лама-3.1-70Б** без каких-либо дополнительных хитростей,
- **Лама-3.1-8Б** со всеми дополнительными трюками, которые вы можете использовать.
Сможете ли вы достичь качества **Llama-3.1-70B** с **Llama-3.1-8B**, оставаясь при этом дешевле?

In [ ]:
# <Your experiments here>

# Практика, часть 3. PRM и поиск луча

## Использование ПРМ
В этом разделе вы будете играть с **PRM** (**Модель процесса вознаграждения**).
На данный момент нам неизвестно ни о каком хорошем API PRM (если вы знаете его, поделитесь с нами!), поэтому нам придется использовать модель с открытым исходным кодом. А именно, мы будем использовать этот: [RLHFlow/Llama3.1-8B-PRM-Deepseek-Data](https://huggingface.co/RLHFlow/Llama3.1-8B-PRM-Deepseek-Data), который является улучшенной версией Llama-3.1-8B. Мы еще не обсуждали использование моделей с открытым исходным кодом, но у вас будет весь код, поэтому мы надеемся, что у вас не возникнет с ним проблем.
**Для запуска этой модели вам понадобится графический процессор**. Сама модель займет около 16 ГБ, а для вывода будет использовано еще немного памяти графического процессора. Для этой задачи должно хватить L40 в облаке Nebius или L4 в Colab. Только не забудьте после завершения выключить (а лучше удалить) виртуальную машину; в противном случае с вас будет взиматься плата за время простоя.

Мы создали класс-оболочку `ProcessRewardModel`, который загружает модель за вас, а также предлагает метод `evaluate_partial_solution(prompt, partial solution)`, который инкапсулирует все детали вызова модели, позволяя вам не думать о том, как это делается «под капотом».
Однако, если вам интересно, мы немного расскажем о том, как работает эта модель. Это модель чата, и на каждое сообщение пользователя он обучен отвечать с помощью `"+"` или `"-"`. Вероятность предсказания `"+"` — это в точности оценка частичного решения («насколько вероятно, что оно даст правильное решение, если продолжить»).
Многоэтапное решение, оцененное с помощью этого PRM, преобразуется в такой диалог:
```
[
      {"role": "user", "content": "Convert the point $(0,3)$ in rectangular coordinates to polar coordinates. To convert from rectangular coordinates $(x, y)$ to polar coordinates $(r, \\theta)$, we can use the formulas\n\\[r = \\sqrt{x^2 + y^2}\\]\n\\[\\theta = \\arctan \\frac{y}{x}\\]"},
      {"role": "assistant", "content": "+"},
      {"role": "user", "content": "In this case, the rectangular coordinates are $(0,3)$, so $x = 0$ and $y = 3$."},
      {"role": "assistant", "content": "+"},
      {"role": "user", "content": "First, we calculate $r$:\n\\[r = \\sqrt{0^2 + 3^2} = \\sqrt{9} = 3\\]"},
      {"role": "assistant", "content": "+"},
      {"role": "user", "content": "Next, we calculate $\\theta$:\n\\[\\theta = \\arctan \\frac{3}{0}\\]"},
      {"role": "assistant", "content": "+"},
      {"role": "user", "content": "Since the tangent function is not defined for $x = 0$, we need to use a special case. When $x = 0$, $\\theta = \\frac{\\pi}{2}$ if $y > 0$, and $\\theta = \\frac{3\\pi}{2}$ if $y < 0$."},
      {"role": "assistant", "content": "+"},
      {"role": "user", "content": "In this case, $y = 3 > 0$, so $\\theta = \\frac{\\pi}{2}$."},
      {"role": "assistant", "content": "+"},
      {"role": "user", "content": "So, the polar coordinates equivalent to $(0,3)$ are $\\boxed{(3,\\frac{\\pi}{2})}$."},
      {"role": "assistant", "content": "+"},
]
```
Источник: [github модели](https://github.com/RLHFlow/RLHF-Reward-Modeling/tree/main/math-rm)
По сути, `evaluate_partial_solution` делает следующее:
* Разделяет решение на `"\n\n"` (конец абзаца), что, по-видимому, является показателем изменения «мыслей» в обучении PRM,
* Превращает шаги решения в диалог.
* Возвращает оценку вероятности создания `"+"`.
Код ниже в основном представляет собой адаптацию [этого оценочного сценария](https://github.com/RLHFlow/RLHF-Reward-Modeling/blob/main/math-rm/prm_evaluate.py).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class ProcessRewardModel:
    def __init__(self, model: str = "RLHFlow/Llama3.1-8B-PRM-Deepseek-Data"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model)
        self.model = AutoModelForCausalLM.from_pretrained(
            model,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto"
        )

        # Set up tokenizer settings
        self.tokenizer.padding_side = "right"
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model.config.pad_token_id = self.model.config.eos_token_id

        # Get token IDs for + and -
        self.plus_token_id = self.tokenizer.encode("+")[-1]
        self.minus_token_id = self.tokenizer.encode("-")[-1]
        self.candidate_tokens = [self.plus_token_id, self.minus_token_id]

    def evaluate_partial_solution(self, prompt: str, partial_solution: str) -> float:
        """Evaluate a partial solution using PRM."""
        # Split solution into steps
        steps = [step.strip() for step in partial_solution.split("\n\n") if step.strip()]

        # Convert to chat format starting with prompt
        conversation = []
        first_text = prompt + " " + steps[0]
        conversation.append({"role": "user", "content": first_text})
        conversation.append({"role": "assistant", "content": "+"})

        for step in steps[1:]:
            conversation.append({"role": "user", "content": step})
            conversation.append({"role": "assistant", "content": "+"})

        # Remove last assistant message for scoring
        conversation = conversation[:-1]

        # Get model prediction
        input_ids = self.tokenizer.apply_chat_template(
            conversation,
            return_tensors="pt"
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(input_ids)
            logits = outputs.logits[:, -3, self.candidate_tokens]
            scores = logits.softmax(dim=-1)
            plus_prob = scores[:, 0].item()  # Probability of + token

        return plus_prob

Загрузим ПРМ и зачтем с его помощью одно правильное и одно неправильное решение уравнения $x^2 - x - 2 = 0$.

In [ ]:
# Initialize PRM once
prm = ProcessRewardModel(model="RLHFlow/Llama3.1-8B-PRM-Deepseek-Data")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/896 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
prompt = "Solve the equation $x^2 - x - 2 = 0$."

# A correct solution
solution = """Factor the left part as $(x + 1)(x - 2)$.

Now, let's rewrite the initial equation $x^2 - x - 2 = 0$ as $(x + 1)(x - 2) = 0$.

Thus, either $x + 1 = 0$ or $x - 2 = 0$.

Thus, the roots are -1 and 2"""

# Get step-score pairs
steps = solution.split("\n\n")
for i in range(len(steps)):
    partial_solution = "\n\n".join(steps[:i+1])
    step_score = prm.evaluate_partial_solution(prompt, partial_solution)
    print(f"\n#PARTIAL SOLUTION:\n {partial_solution}")
    print(f"#SCORE: {step_score:.4f}")


#PARTIAL SOLUTION:
 Factor the left part as $(x + 1)(x - 2)$.
#SCORE: 0.9995

#PARTIAL SOLUTION:
 Factor the left part as $(x + 1)(x - 2)$.

Now, let's rewrite the initial equation $x^2 - x - 2 = 0$ as $(x + 1)(x - 2) = 0$.
#SCORE: 0.9478

#PARTIAL SOLUTION:
 Factor the left part as $(x + 1)(x - 2)$.

Now, let's rewrite the initial equation $x^2 - x - 2 = 0$ as $(x + 1)(x - 2) = 0$.

Thus, either $x + 1 = 0$ or $x - 2 = 0$.
#SCORE: 0.9209

#PARTIAL SOLUTION:
 Factor the left part as $(x + 1)(x - 2)$.

Now, let's rewrite the initial equation $x^2 - x - 2 = 0$ as $(x + 1)(x - 2) = 0$.

Thus, either $x + 1 = 0$ or $x - 2 = 0$.

Thus, the roots are -1 and 2
#SCORE: 1.0000


Мы кратко прокомментируем вывод выше. Решение состоит из 4 абзацев и, таким образом, разделено на 4 отдельные «мысли» (этапы решения). В цикле мы оцениваем:
* Шаг 1
* Шаги 1+2
* Шаги 1+2+3
* Шаги 1+2+3+4 (полное решение)
Как видите, все оценки довольно высокие, а полное решение получает высшую оценку (и это действительно правильно).

In [ ]:
prompt = "Solve the equation $x^2 - x - 2 = 0$."

# An incorrect solution
solution = """Rewrite the equation as $x^2 = x + 2$.

Divide both parts by x: $x = 1 + 2x$.

Rewrite it as $x = 1$. Thus, x = 1."""

# Get step-score pairs
steps = solution.split("\n\n")
for i in range(len(steps)):
    partial_solution = "\n\n".join(steps[:i+1])
    step_score = prm.evaluate_partial_solution(prompt, partial_solution)
    print(f"\n#PARTIAL SOLUTION:\n {partial_solution}")
    print(f"#SCORE: {step_score:.4f}")


#PARTIAL SOLUTION:
 Rewrite the equation as $x^2 = x + 2$.
#SCORE: 0.5889

#PARTIAL SOLUTION:
 Rewrite the equation as $x^2 = x + 2$.

Divide both parts by x: $x = 1 + 2x$.
#SCORE: 0.1550

#PARTIAL SOLUTION:
 Rewrite the equation as $x^2 = x + 2$.

Divide both parts by x: $x = 1 + 2x$.

Rewrite it as $x = 1$. Thus, x = 1.
#SCORE: 0.3999


За неправильное решение баллы значительно ниже. Более того:
* Первый шаг, несколько странный при решении простого квадратного уравнения, оценивается как сомнительный.
* Следующий шаг, приводящий к серьезной ошибке, получает очень низкую оценку.
Итак, PRM, похоже, в некоторой степени соответствует нашей математической интуиции.

## Поиск луча
В этой части мы поделимся нашей реализацией **Beam Search** — классом `MathBeamSearch`. Он довольно большой, поэтому прокомментируем несколько важных моментов:
* Он использует [min heap](https://en.wikipedia.org/wiki/Heap_(data_structure)) для хранения вновь сгенерированных завершений и их оценок, поскольку эта структура данных позволяет быстро добавлять и быстро удалять минимальные элементы.
  Мы выбрали реализацию `heapq`; поскольку `heapq` — это максимальная куча, мы фактически храним пары `(-score, partial_solution)` для создания из нее минимальной кучи.
* Каждый раз LLM предлагается сгенерировать следующий логический шаг решения и сохранить его в одной строке. Это (в большинстве случаев) позволяет установить `"\n\n"` в качестве разделителей между отдельными «мыслями», как и ожидалось PRM.
* Мы предлагаем LLM вывести `#ANSWER: <answer>` когда он получит окончательный ответ. Это позволяет дорабатывать успешные решения, не продолжая их бесцельно, пока они не достигнут `max_steps` «мыслей».

In [ ]:
import heapq
from typing import List, Dict, Tuple, Optional
from openai import OpenAI

class LLMClient:
    """Wrapper for OpenAI-compatible API clients with consistent interface."""
    def __init__(
        self,
        client: OpenAI,
        model: str,
        default_temperature: float = 0.0,
        default_max_tokens: int = 1024,
        system_prompt: Optional[str] = None
    ):
        self.client = client
        self.model = model
        self.default_temperature = default_temperature
        self.default_max_tokens = default_max_tokens
        self.system_prompt = system_prompt

    def generate(
        self,
        prompt: str,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
        system_prompt: Optional[str] = None
    ) -> str:
        """Generate completion with consistent interface across different LLM providers."""
        messages = []

        # Use provided system prompt or fall back to default
        current_system_prompt = system_prompt or self.system_prompt
        if current_system_prompt:
            messages.append({"role": "system", "content": current_system_prompt})

        messages.append({"role": "user", "content": prompt})

        completion = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature if temperature is not None else self.default_temperature,
            max_tokens=max_tokens if max_tokens is not None else self.default_max_tokens
        )

        return completion.choices[0].message.content

class MathBeamSearch:
    """Beam search implementation for math problem solving."""
    def __init__(
        self,
        prm: ProcessRewardModel,
        llm_client: LLMClient,
        beam_width: int = 2,
        max_steps: int = 10
    ):
        self.prm = prm
        self.llm_client = llm_client
        self.beam_width = beam_width
        self.max_steps = max_steps

    def generate_next_steps(self, prompt: str, partial_solution: str, num_continuations: int) -> List[str]:
        """Generate next possible steps using LLM."""
        if partial_solution:
            message = f"""You are an expert math problem solver. Given a math problem and a partial solution, generate the next logical step.
Keep the step concise and focused on one specific calculation or logical deduction.

Problem:
{prompt}

Current partial solution:
{partial_solution}

Generate the next step in the solution. Only output:
- the next step
- #ANSWER: followed by your answer, if you can determine the final answer. If you output #ANSWER:, you need to output the actual answer after it.
Don't output anything else!
Keep the whole new step of the single line.
If you need to write formulas, use latex $ markup to format them."""
        else:
            message = f"""You are an expert math problem solver. Given a math problem, generate the first step of the solution.
Keep the step concise and focused on one specific calculation or logical deduction.

Problem:
{prompt}

Generate the first step in the solution. Only output:
- the first step,
- #ANSWER: followed by your answer, if you can determine the final answer. If you output #ANSWER:, you need to output the actual answer after it.
Don't output anything else!
Keep the whole first step of the single line.
If you need to write formulas, use latex $ markup to format them."""

        responses = []
        for _ in range(num_continuations):
            response = self.llm_client.generate(
                message
            )
            responses.append(response.strip())
        return responses

    def beam_search(self, prompt: str,
            verbose: bool = False) -> List[Tuple[float, str]]:
        """Perform beam search to find the best solution."""
        # Initialize beam with empty solutions
        current_beam = [(0.0, "", False)]  # (score, solution, is_finalized)

        # Get initial steps
        initial_continuations = self.generate_next_steps(prompt, None, self.beam_width)

        # Initialize beam with scored initial steps
        candidates = []
        for continuation in initial_continuations:
            score = self.prm.evaluate_partial_solution(prompt, continuation)
            is_finalized = "#ANSWER:" in continuation
            candidates.append((-score, continuation, is_finalized))  # Negative for max-heap
            if verbose:
                print(f"\nInitial step (score {score:.4f}):")
                print(f"{continuation}\n")

        # Select top-k candidates for initial beam
        heapq.heapify(candidates)
        current_beam = [(-score, solution, is_finalized)
                       for score, solution, is_finalized in heapq.nsmallest(self.beam_width, candidates)]

        # Beam search iterations
        step = 0
        while step < self.max_steps:
            # Check if all solutions are finalized
            if all(is_finalized for _, _, is_finalized in current_beam):
                break

            if verbose:
                print(f"\n=== Step {step + 1} ===")
            candidates = []

            # Keep finalized solutions and generate continuations for unfinished ones
            for score, partial_solution, is_finalized in current_beam:
                if is_finalized:
                    # Keep finalized solutions in candidates without modification
                    candidates.append((-score, partial_solution, True))
                else:
                    # Generate continuations only for unfinished solutions
                    continuations = self.generate_next_steps(
                        prompt,
                        partial_solution,
                        self.beam_width
                    )

                    # Evaluate each continuation
                    for continuation in continuations:
                        new_solution = partial_solution + "\n\n" + continuation if partial_solution else continuation
                        new_score = self.prm.evaluate_partial_solution(prompt, new_solution)
                        is_finished = "#ANSWER:" in continuation
                        candidates.append((-new_score, new_solution, is_finished))
                        if verbose:
                            print(f"\nCandidate (score {new_score:.4f}):")
                            print(f"{continuation}\n")

            # Select top-k candidates for next beam
            heapq.heapify(candidates)
            current_beam = [(-score, solution, is_finalized)
                          for score, solution, is_finalized in heapq.nsmallest(self.beam_width, candidates)]

            if verbose:
                print("\nSelected for next beam:")
                for score, solution, is_finalized in current_beam:
                    status = "FINALIZED" if is_finalized else "IN PROGRESS"
                    print(f"\nScore: {score:.4f} [{status}]")
                    print(f"{solution}\n")

            step += 1

        # Return all solutions (now guaranteed to include any finalized ones)
        return [(score, solution) for score, solution, _ in current_beam]

Давайте попробуем!

In [ ]:
client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)


# Create beam search instance
beam_search = MathBeamSearch(
    prm=prm,
    llm_client=LLMClient(
        client=client,
        model="meta-llama/Meta-Llama-3.1-70B-Instruct",
        default_temperature=1,
        default_max_tokens=8192
    ),
    beam_width=2,
    max_steps=20
)

Включим параметр `verbose`, чтобы увидеть все промежуточные результаты. (По умолчанию это `False` .)

In [ ]:
prompt = "Inside a circle, two parallel chords are 6 units apart. One chord has length 14 and the other has length 10. Find the radius of the circle."

# Run beam search
results = beam_search.beam_search(prompt, verbose=True)

# Print final results
print("\n=== Final Results ===")
for score, solution in results:
    print(f"\nScore: {score:.4f}")
    print(f"{solution}\n")


Initial step (score 0.2830):
Let $r$ be the radius of the circle, and draw a perpendicular line from the center of the circle to each of the two chords, which divides each chord into two equal parts: $7$ and $5$.
$ANSWER:


Initial step (score 0.6875):
Draw a perpendicular line from the center of the circle to each chord, and denote the radius of the circle as $r$. Let $d$ be the distance from the center of the circle to the chord with length 10, so the distance from the center of the circle to the chord with length 14 is $d+6$.


=== Step 1 ===

Candidate (score 0.4688):
Using the Pythagorean theorem, we can set up two equations: $r^2 = d^2 + 5^2$ and $r^2 = (d+6)^2 + 7^2$.


Candidate (score 0.5796):
Apply the Pythagorean theorem to the two right triangles formed by the radii and the chords, resulting in the equations: $r^2 = d^2 + 5^2$ and $r^2 = (d+6)^2 + 7^2$.


Candidate (score 0.8408):
The perpendicular line from the center of the circle and the chords form two right triangles:

# Практика, часть 4: Уверенность как синтетический PRM
Обучение **Моделям процесса вознаграждения (PRM)** является сложной задачей, и на Hugging Face доступно лишь несколько таких моделей, и ни одна из них не идеальна. Поэтому было бы полезно иметь **безмодельный** метод оценки решений. Одним из простых суррогатов вознаграждения за процесс, который следует учитывать, является **уверенность**.
В [Блокноте «Параметры вывода LLM](https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic1/1.6_llm_inference_parameters.ipynb), мы обсуждали, что LLM демонстрируют разные уровни уверенности в своих сгенерированных результатах:
<center>
<img src="https://drive.google.com/uc?export=view&id=12k5EFzMZAcHntuJZBZwbm6NKqJZ1OF3l" width=600 />
</center>
Левое изображение иллюстрирует случай, когда LLM почти наверняка сгенерирует «LLM», а правое изображение показывает сценарий, когда модель менее уверена в своих результатах. Хотя неопределенность может быть ценной в творческом письме, она может указывать на путаницу или даже галлюцинации при решении математических задач. Таким образом, для задач по математике и логическому рассуждению разумно предположить, что **решения, полученные с более высокой степенью достоверности, с большей вероятностью будут правильными**.
### Простой подход: использование максимальной прогнозируемой вероятности
Учитывая это, мы предлагаем изменить алгоритм **Beam Search** для оценки частичных решений на основе их **средней достоверности**. Достоверность можно оценить с помощью **средней максимальной прогнозируемой логарифмической вероятности**, рассчитываемой как:
$$\frac{1}{\mathrm{n\_steps}}\sum_{i=1}^{\mathrm{n\_steps}}\log\left(\mbox{Вероятность максимального токена, предсказанная на шаге $i$}\right)$$
Максимальную вероятность можно получить, вызвав `client.chat.completions.create` с помощью `logprobs=True` и извлекая `completion.choices[0].logprobs`.
Хотя этот подход довольно упрощен, он все же может быть эффективным. Более высокая максимальная вероятность подразумевает более низкие вероятности для альтернативных токенов, что указывает на большую уверенность в верхнем прогнозе.
### Более интересный подход: отрицательная средняя энтропия
Более надежный метод предполагает использование **отрицательной средней энтропии**. [Энтропия](https://en.wikipedia.org/wiki/Entropy_(information_theory)) количественно определяет неопределенность распределения вероятностей. Для следующего поколения токена она рассчитывается как:
$$-\sum_{w\in\mbox{Vocab}}\widehat{p}_{w}\log{\widehat{p}_{w} },$$
где $\widehat{p}_{w}$ представляет собой прогнозируемую вероятность токена $w$. Энтропия ведет себя следующим образом:
- $0$, когда вероятность одного токена равна 1, а вероятность всех остальных равна 0 (**абсолютная уверенность**).
- Максимум, когда все токены имеют равные вероятности (**абсолютная неопределенность**).
Таким образом, решения с **более низкой энтропией** генерируются более уверенно.
К сожалению, API OpenAI предоставляет только топ-5 вероятностей токена, что ограничивает прямой расчет энтропии. Однако энтропию все же можно оценить, используя эти пять главных вероятностей. Итак, вы тоже можете попробовать это, но мы рекомендуем вам начать с использования только максимальной вероятности.

In [ ]:
# <YOUR CODE HERE>